In [ ]:
!pip install --no-index \
    -r /kaggle/input/notebooks/ttahara/birdclef-2026-download-wheels/requirements.txt \
    --find-links=/kaggle/input/notebooks/ttahara/birdclef-2026-download-wheels/wheels

In [ ]:
K = 2

# 5s windows
### 2026 HgnetV2-B0(no pseudo) fold0
### 2026 EfficientnetV2-S(1 iter pseudo) fold0
### 2026 Efficientnetb3(1 iter pseudo) fold0

In [ ]:
import gc
import os
import re
import subprocess, json, time
import tempfile
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile
import torch
import torchaudio
from torchvision.transforms import v2 as tvt_v2
import torch.nn.functional as F
from torch import nn
import timm
import openvino as ov
from tqdm.notebook import tqdm, trange

ROOT = Path.cwd().parent
INPUT = ROOT / "input"
DATA = INPUT / "competitions" / "birdclef-2026"
TEST_SS = DATA / "test_soundscapes"
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
taxonomy = pd.read_csv(DATA / "taxonomy.csv")
CLASSES = taxonomy.primary_label.values.tolist()
N_CLASSES = 234
N_WINDOWS = 12
USE_ADAPT_SMOOTH = False
ADAPT_ALPHA = 0.20
HGNET_PATH_A = "/kaggle/input/datasets/jakkma/hgnetv2b0-fold0/hgnetv2b0_seed520.pt"
HGNET_PATH_B = "/kaggle/input/datasets/jakkma/hgnetv2b0-fold0/hgnetv2b0_seed3407.pt"
EFF_PATH_A   = "/kaggle/input/datasets/jakkma/pseudo-efficientnetv2s-fold0/efficientnetv2s_seed1086_5s.pt"  
EFF_PATH_B   = "/kaggle/input/datasets/jakkma/pseudo-efficientnetv2s-fold0/efficientnetv2s_seed42_5s.pt"
EFF_B3 = "/kaggle/input/datasets/jakkma/efficientnetb3/efficientnetb3_seed1086_iter1.pt"
SAMPLE_RATE = 32_000
MAX_SEC = 60
DURATION = SAMPLE_RATE * MAX_SEC
BATCH_SIZE = 12
MEL_PARAMS_HGNET = dict(
    sample_rate=32000, n_fft=2048, hop_length=512,
    f_min=20, f_max=16000, n_mels=128, normalized=True,
)
MEL_PARAMS_EFF = dict(
    sample_rate=32000, n_fft=2048, hop_length=512,
    f_min=0, f_max=16000, n_mels=128, normalized=True,
)
MEL_PARAMS_EFFB3 = dict(
    sample_rate=32000, n_fft=3072, hop_length=420,
    f_min=0, f_max=16000, n_mels=384, normalized=True,
)
LMS_SHAPE = (128, 313)
TOP_DB = 80.0

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def fuse_tta(norm_by_file, shift_by_file):
    out = np.zeros_like(norm_by_file)
    out[:, 0, :] = 0.75 * norm_by_file[:, 0, :] + 0.25 * shift_by_file[:, 0, :]
    out[:, 11, :] = 0.75 * norm_by_file[:, 11, :] + 0.25 * shift_by_file[:, 10, :]
    out[:, 1:11, :] = (0.50 * norm_by_file[:, 1:11, :] + 0.25 * shift_by_file[:, 0:10, :] + 0.25 * shift_by_file[:, 1:11, :])
    return out

def init_layer(layer):
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, "bias") and layer.bias is not None:
        layer.bias.data.fill_(0.)

def init_bn(bn):
    bn.bias.data.fill_(0.)
    bn.weight.data.fill_(1.0)

class GeM1d(nn.Module):
    def __init__(self, p=3.0, kernel_size=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.kernel_size = kernel_size
        self.eps = eps
    def forward(self, x):
        x = x.clamp(min=self.eps)
        return F.avg_pool1d(x.pow(self.p), kernel_size=self.kernel_size,
                            stride=1, padding=self.kernel_size//2).pow(1.0/self.p)

class ChannelAttention(nn.Module):
    def __init__(self, dim=2048, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(dim, dim//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(dim//reduction, dim, bias=False))
        self.sigmoid = nn.Sigmoid()
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x).squeeze(-1))
        max_out = self.fc(self.max_pool(x).squeeze(-1))
        out = self.sigmoid(avg_out + max_out).unsqueeze(-1)
        return x * out

class FrequencySE(nn.Module):
    def __init__(self, n_mels=128, reduction=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d((n_mels, 1))
        self.fc = nn.Sequential(
            nn.Linear(n_mels, n_mels//reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(n_mels//reduction, n_mels, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        b, c, h, w = x.size()
        y = self.avg_pool(x).squeeze(-1).squeeze(1)
        y = self.fc(y).view(b, 1, h, 1)
        return x * y.expand_as(x)

class AttBlock_Topk(nn.Module):
    def __init__(self, in_features, out_features, k=3, r=1.0):
        super().__init__()
        self.att = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.cla = nn.Conv1d(in_features, out_features, kernel_size=1, bias=True)
        self.bn_att = nn.BatchNorm1d(out_features)
        init_layer(self.att)
        init_layer(self.cla)
        init_bn(self.bn_att)
        self.k = k
    def forward(self, x):
        norm_att = torch.softmax(torch.clamp(self.att(x), -10, 10), dim=-1)
        framewise = self.cla(x)
        clipwise_att = torch.sum(norm_att * framewise, dim=2)
        actual_k = min(self.k, framewise.size(-1))
        clipwise_aux = framewise.topk(actual_k, dim=2)[0].mean(dim=2)
        return clipwise_att
class AttBlock(nn.Module):
    def __init__(self, in_feat, out_feat):
        super().__init__()
        self.att = nn.Conv1d(in_feat, out_feat, 1, bias=True)
        self.cla = nn.Conv1d(in_feat, out_feat, 1, bias=True)
        init_layer(self.att)
        init_layer(self.cla)

    def forward(self, x):
        norm_att = torch.softmax(torch.clamp(self.att(x), -10, 10), dim=-1)
        framewise = self.cla(x)
        clipwise = torch.sum(norm_att * framewise, dim=2)
        return clipwise
        
class DistillHead(nn.Module):
    def __init__(self, backbone_dim, embed_dim=1536):
        super().__init__()
        self.proj = nn.Linear(backbone_dim, embed_dim)
        init_layer(self.proj)
    def forward(self, feat_map):
        return self.proj(feat_map.mean(dim=[2,3]))

class HgnetModel(nn.Module):
    def __init__(self, model_name, pretrained=False, drop_path_rate=0.20, drop_rate=0.20,
                 num_classes=234, head_dropout=0.5, n_mels=128):
        super().__init__()
        self.bn0 = nn.BatchNorm2d(n_mels)
        init_bn(self.bn0)
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, in_chans=1,
            global_pool="", num_classes=0,
            drop_path_rate=drop_path_rate, drop_rate=drop_rate,
        )
        with torch.no_grad():
            n_feat = self.backbone(torch.randn(1, 1, n_mels, n_mels)).shape[1]
        self.gem = GeM1d(p=3.0, kernel_size=3)
        self.fc1 = nn.Linear(n_feat, n_feat, bias=True)
        self.att_block = AttBlock_Topk(n_feat, num_classes)
        self.dropout = nn.Dropout(head_dropout)
        self.distill_head = DistillHead(n_feat, 1536)
        self.channel_att = ChannelAttention()
        self.freq_se = FrequencySE(n_mels=n_mels, reduction=16)
        init_layer(self.fc1)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.bn0(x)
        x = x.transpose(1, 2)
        x = self.freq_se(x)
        feat = self.backbone(x)
        feat = feat.mean(dim=2)
        feat = self.channel_att(feat)
        feat = self.gem(feat)
        feat = self.dropout(feat)
        feat = feat.transpose(1, 2)
        feat = F.relu(self.fc1(feat))
        feat = feat.transpose(1, 2)
        feat = self.dropout(feat)
        clipwise = self.att_block(feat)
        return clipwise

class EffNetModel(nn.Module):
    def __init__(self, model_name, pretrained=False, drop_path_rate=0.15, drop_rate=0.2,
                 num_classes=234, head_dropout=0.35, n_mels=128):
        super().__init__()
        self.bn0 = nn.BatchNorm2d(n_mels)
        init_bn(self.bn0)
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained, in_chans=1,
            global_pool="", num_classes=0,
            drop_path_rate=drop_path_rate, drop_rate=drop_rate,
        )
        with torch.no_grad():
            n_feat = self.backbone(torch.randn(1, 1, n_mels, n_mels)).shape[1]
        self.gem = GeM1d(p=3.0, kernel_size=3)
        self.fc1 = nn.Linear(n_feat, n_feat, bias=True)
        self.att_block = AttBlock(n_feat, num_classes)
        self.dropout = nn.Dropout(head_dropout)
        self.distill_head = DistillHead(n_feat, 1536)
        init_layer(self.fc1)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.bn0(x)
        x = x.transpose(1, 2)
        feat = self.backbone(x)
        feat = feat.mean(dim=2)
        feat = self.gem(feat)
        feat = self.dropout(feat)
        feat = feat.transpose(1, 2)
        feat = F.relu(self.fc1(feat))
        feat = feat.transpose(1, 2)
        feat = self.dropout(feat)
        clipwise = self.att_block(feat)
        return clipwise

def ov_model_from_pt(model_class, pt_path, model_kwargs, device="CPU", input_shape=(1,1,128,313)):
    model = model_class(**model_kwargs)
    m, u = model.load_state_dict(torch.load(pt_path, map_location="cpu"), strict=False)
    print(f"Loaded {pt_path}: missing={m}, unexpected={u}")
    model.eval()
    class InferenceWrapper(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model
        def forward(self, x):
            return self.base_model(x)
    export_model = InferenceWrapper(model)
    export_model.eval()
    dummy = torch.randn(*input_shape, dtype=torch.float32)
    with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as f:
        onnx_path = f.name
    try:
        torch.onnx.export(export_model, dummy, onnx_path,
                          opset_version=14, do_constant_folding=False, dynamo=False,
                          input_names=["input"], output_names=["output"],
                          dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}})
        ov_model = ov.convert_model(onnx_path)
    finally:
        os.unlink(onnx_path)
    compiled = ov.compile_model(ov_model, device, {
        "PERFORMANCE_HINT": "THROUGHPUT",
        "INFERENCE_NUM_THREADS": 4,
        "NUM_STREAMS": 2,
    })
    return compiled

def infer_stream(engine, wave_batches, lms_transform, num_segments, num_classes):
    logits = np.zeros((num_segments, num_classes), dtype=np.float32)
    infer_queue = ov.AsyncInferQueue(engine, 16)
    input_name = engine.inputs[0].get_any_name()
    def callback(request, userdata):
        logits[userdata] = request.get_output_tensor().data
    infer_queue.set_callback(callback)
    total = 0
    for waves in tqdm(wave_batches, desc="Inference"):
        bsize = len(waves)
        idxs = np.arange(total, total + bsize)
        total += bsize
        lms = lms_transform(torch.from_numpy(waves)).numpy().astype(np.float32)
        infer_queue.start_async({input_name: lms}, userdata=idxs)
    infer_queue.wait_all()
    return logits

def load_and_prepare_audio(test_ss_paths):
    wave_list = []
    shifted_wave_list = []
    for path in tqdm(test_ss_paths, desc="Loading Audio"):
        with soundfile.SoundFile(path) as f:
            audio_data = f.read(frames=DURATION, dtype="float32")
            if audio_data.ndim > 1:
                audio_data = audio_data.mean(axis=1)
            if len(audio_data) < DURATION:
                audio_data = np.pad(audio_data, (0, DURATION - len(audio_data)))
        for seg_sec in range(0, MAX_SEC, 5):
            start = seg_sec * SAMPLE_RATE
            end = (seg_sec + 5) * SAMPLE_RATE
            wave_list.append(audio_data[start:end])
        for seg_sec in range(0, MAX_SEC - 5, 5):
            start = int((seg_sec + 2.5) * SAMPLE_RATE)
            end = int((seg_sec + 7.5) * SAMPLE_RATE)
            shifted_wave_list.append(audio_data[start:end])
    return wave_list, shifted_wave_list

def rank_normalize(x):
    r_x = np.zeros_like(x)
    for i in range(x.shape[1]):
        r_x_i = pd.Series(x[:, i]).rank(method="max")
        r_x[:, i] = r_x_i / r_x_i.shape[0]
    return r_x


IS_TEST_ENV = len(sample_sub) > 10
if IS_TEST_ENV:
    test_ss_paths = []
    added = set()
    for row_id in sample_sub["row_id"].values:
        file_id = "_".join(row_id.split("_")[:-1])
        if file_id in added:
            continue
        added.add(file_id)
        test_ss_paths.append(TEST_SS / f"{file_id}.ogg")
else:
    dry_n = K
    test_ss_paths = sorted((DATA / "train_soundscapes").iterdir())[:dry_n]
test_ss_segs = []
for p in test_ss_paths:
    for i in range(0, 60, 5):
        test_ss_segs.append(f"{p.stem}_{i+5}")

wave_list, shifted_wave_list = load_and_prepare_audio(test_ss_paths)
num_test_ss_audios = len(test_ss_paths)
num_test_ss_segs = len(wave_list)
num_shifted_test_ss_segs = len(shifted_wave_list)
print(f"Normal segments: {num_test_ss_segs}")
print(f"Shifted segments: {num_shifted_test_ss_segs}")

wave_batches = [np.stack(wave_list[i:i+BATCH_SIZE], axis=0)
                    for i in trange(0, len(wave_list), BATCH_SIZE, desc="Stack Normal")]
shifted_wave_batches = [np.stack(shifted_wave_list[i:i+BATCH_SIZE], axis=0)
                            for i in trange(0, len(shifted_wave_list), BATCH_SIZE, desc="Stack Shifted")]
del wave_list, shifted_wave_list
gc.collect()

lms_hgnet = tvt_v2.Compose([
    torchaudio.transforms.MelSpectrogram(**MEL_PARAMS_HGNET),
    torchaudio.transforms.AmplitudeToDB(stype="power", top_db=TOP_DB),
    tvt_v2.Resize(size=LMS_SHAPE),
    lambda x: torch.clamp((x + TOP_DB) / TOP_DB, 0.0, 1.0),
    lambda x: x[:, None, :, :]
])
lms_eff = tvt_v2.Compose([
    torchaudio.transforms.MelSpectrogram(**MEL_PARAMS_EFF),
    torchaudio.transforms.AmplitudeToDB(stype="power", top_db=TOP_DB),
    tvt_v2.Resize(size=LMS_SHAPE),
    lambda x: torch.clamp((x + TOP_DB) / TOP_DB, 0.0, 1.0),
    lambda x: x[:, None, :, :]
])
lms_effb3 = tvt_v2.Compose([
    torchaudio.transforms.MelSpectrogram(**MEL_PARAMS_EFFB3),
    torchaudio.transforms.AmplitudeToDB(stype="power", top_db=TOP_DB),
    tvt_v2.Resize(size=(384, 384)),
    lambda x: torch.clamp((x + TOP_DB) / TOP_DB, 0.0, 1.0),
    lambda x: x[:, None, :, :]
])
class TransformWrapper(nn.Module):
    def __init__(self, transform):
        super().__init__()
        self.transform = transform
    @torch.no_grad()
    def forward(self, wave):
        return self.transform(wave)
lms_hgnet = TransformWrapper(lms_hgnet).eval()
lms_eff = TransformWrapper(lms_eff).eval()
lms_effb3 = TransformWrapper(lms_effb3).eval()

hgnet_kwargs = dict(
        model_name="hgnetv2_b0.ssld_stage2_ft_in1k",
        pretrained=False, drop_path_rate=0.15, drop_rate=0.20,
        num_classes=N_CLASSES, head_dropout=0.45, n_mels=128,
)
eff_kwargs = dict(
        model_name="tf_efficientnetv2_s.in21k",
        pretrained=False, drop_path_rate=0.15, drop_rate=0.2,
        num_classes=N_CLASSES, head_dropout=0.35, n_mels=128,
)
effb3_kwargs = dict(
        model_name="tf_efficientnet_b3.ns_jft_in1k",
        pretrained=False, drop_path_rate=0.05, drop_rate=0.1,
        num_classes=N_CLASSES, head_dropout=0.2, n_mels=384,
)

engine_hgnet_A = ov_model_from_pt(HgnetModel, HGNET_PATH_A, hgnet_kwargs)
engine_eff_A = ov_model_from_pt(EffNetModel, EFF_PATH_A, eff_kwargs)
engine_eff_b3 = ov_model_from_pt(EffNetModel, EFF_B3, effb3_kwargs, input_shape=(1,1,384,384))

logits_h_norm_A = infer_stream(engine_hgnet_A, wave_batches, lms_hgnet, num_test_ss_segs, N_CLASSES)
logits_h_shift_A = infer_stream(engine_hgnet_A, shifted_wave_batches, lms_hgnet, num_shifted_test_ss_segs, N_CLASSES)
pred_h_norm_A = sigmoid(logits_h_norm_A).reshape(num_test_ss_audios, 12, N_CLASSES)
pred_h_shift_A = sigmoid(logits_h_shift_A).reshape(num_test_ss_audios, 11, N_CLASSES)
tta_h_cross1 = fuse_tta(pred_h_norm_A, pred_h_shift_A)
final_h = tta_h_cross1
final_h_flat = final_h.reshape(num_test_ss_segs, N_CLASSES)

# 平滑函数 (5th 2025)
def temporal_smooth_on_probs(probs, alpha=0.1):
    N, C = probs.shape
    n_files = N // 12
    probs_reshaped = probs.reshape(n_files, 12, C)
    smoothed = probs_reshaped.copy()
    for i in range(1, 11):
        smoothed[:, i, :] = (alpha * probs_reshaped[:, i-1, :] +
                             (1-2*alpha) * probs_reshaped[:, i, :] +
                             alpha * probs_reshaped[:, i+1, :])
    smoothed[:, 0, :] = (1-alpha) * probs_reshaped[:, 0, :] + alpha * probs_reshaped[:, 1, :]
    smoothed[:, 11, :] = (1-alpha) * probs_reshaped[:, 11, :] + alpha * probs_reshaped[:, 10, :]
    return smoothed.reshape(N, C)

final_h_flat = temporal_smooth_on_probs(final_h_flat, alpha=0.1)

del engine_hgnet_A, logits_h_norm_A, logits_h_shift_A
gc.collect()

logits_e_norm_A = infer_stream(engine_eff_A, wave_batches, lms_eff, num_test_ss_segs, N_CLASSES)
logits_e_shift_A = infer_stream(engine_eff_A, shifted_wave_batches, lms_eff, num_shifted_test_ss_segs, N_CLASSES)
pred_e_norm_A = sigmoid(logits_e_norm_A).reshape(num_test_ss_audios, 12, N_CLASSES)
pred_e_shift_A = sigmoid(logits_e_shift_A).reshape(num_test_ss_audios, 11, N_CLASSES)
tta_e_cross1 = fuse_tta(pred_e_norm_A, pred_e_shift_A)
final_e = tta_e_cross1
final_e_flat = final_e.reshape(num_test_ss_segs, N_CLASSES)

# effv2s 应用文件峰值缩放 postprocess (2th 2025) 
def apply_file_peak_scale(probs, top=1):
    N, C = probs.shape
    n_files = N // 12
    probs_reshaped = probs.reshape(n_files, 12, C)
    topk_vals = np.sort(probs_reshaped, axis=1)[:, -top:, :]  
    mean_ = np.mean(topk_vals, axis=1, keepdims=True)       
    scaled = probs_reshaped * mean_
    return scaled.reshape(N, C)
    
final_e_flat = apply_file_peak_scale(final_e_flat, top=1)

del engine_eff_A
gc.collect()

logits_eb3_norm_A = infer_stream(engine_eff_b3, wave_batches, lms_effb3, num_test_ss_segs, N_CLASSES)
final_eb3 = sigmoid(logits_eb3_norm_A)

del engine_eff_b3, lms_effb3, wave_batches, shifted_wave_batches
gc.collect()

# 10s windows
### EfficientnetV2-S (no pseudo) fold0

In [ ]:
import soundfile as sf
class LogMelSpectrogramTransform(nn.Module):
    def __init__(self, mel_params: dict, top_db: float):
        super().__init__()
        self.mel_transform = torchaudio.transforms.MelSpectrogram(**mel_params)
        self.db = torchaudio.transforms.AmplitudeToDB(stype="power", top_db=top_db)
        self.top_db = top_db
    @torch.no_grad()
    def forward(self, wave: torch.Tensor) -> torch.Tensor:
        mel = self.mel_transform(wave)
        lms = self.db(mel)

        lms = torch.clamp((lms + self.top_db) / self.top_db, 0.0, 1.0)
        return lms[:, None, :, :]

def build_context_windows(test_ss_paths, sample_rate=32000, duration_sec=5, ctx_sec=2.5, max_sec=60):
    seg_samples = int(sample_rate * duration_sec)          
    context_samples = int(sample_rate * ctx_sec)          
    win_samples = seg_samples + 2 * context_samples   
    num_segs = max_sec // duration_sec  

    wave_list = []
    for path in tqdm(test_ss_paths, desc="Loading 10s context windows"):
        with sf.SoundFile(path) as f:
            total_frames = min(f.frames, sample_rate * max_sec)
            audio = f.read(frames=total_frames, dtype='float32')
            if audio.ndim > 1:
                audio = audio.mean(axis=1)
        needed = sample_rate * max_sec
        if len(audio) < needed:
            audio = np.pad(audio, (0, needed - len(audio)))
        for i in range(num_segs):
            center_start = i * seg_samples
            win_start = center_start - context_samples
            win_end = center_start + seg_samples + context_samples

            if win_start < 0:
                segment = np.pad(audio[:win_end], (-win_start, 0))
            elif win_end > len(audio):
                segment = np.pad(audio[win_start:], (0, win_end - len(audio)))
            else:
                segment = audio[win_start:win_end]

            if len(segment) < win_samples:
                segment = np.pad(segment, (0, win_samples - len(segment)))
            elif len(segment) > win_samples:
                segment = segment[:win_samples]

            wave_list.append(segment)

    num_audios = len(test_ss_paths)
    num_segments = len(wave_list)
    batch_size = 12
    wave_batches = [np.stack(wave_list[i:i+batch_size], axis=0)
                    for i in trange(0, num_segments, batch_size, desc="Stack windows")]
    del wave_list
    gc.collect()
    return wave_batches, num_segments, num_audios

wave_batches, num_segments, num_audios = build_context_windows(test_ss_paths)
lms_transform = LogMelSpectrogramTransform(MEL_PARAMS_EFF, TOP_DB).eval()
MODEL_10S_A = "/kaggle/input/datasets/jakkma/efficientnetv2s-ctx10s/10s_seed1086.pt"
engine_10s_A = ov_model_from_pt(EffNetModel, MODEL_10S_A, model_kwargs={"model_name": "tf_efficientnetv2_s.in21k"}, input_shape=(1, 1, 128, 626))
logits_A = infer_stream(engine_10s_A, wave_batches, lms_transform, num_segments, N_CLASSES)
final_eff10s = sigmoid(logits_A)

final_eff10s = apply_file_peak_scale(final_eff10s, top=1)  

del wave_batches, engine_10s_A, lms_transform, logits_A
gc.collect()

In [ ]:
hg_df = pd.DataFrame(final_h_flat, columns=CLASSES)
ef_df = pd.DataFrame(final_e_flat, columns=CLASSES)
ef10s_df = pd.DataFrame(final_eff10s, columns=CLASSES)
eb3_df = pd.DataFrame(final_eb3, columns=CLASSES)

rank_hgnet = hg_df.rank(axis=0, pct=True).to_numpy(np.float32)
rank_eff   = ef_df.rank(axis=0, pct=True).to_numpy(np.float32)
rank_eff10 = ef10s_df.rank(axis=0, pct=True).to_numpy(np.float32)
rank_ef3 = eb3_df.rank(axis=0, pct=True).to_numpy(np.float32)

W_HGNET = 0.250
W_EFF   = 0.400
W_EFF10 = 0.250
W_EFB3  = 0.100

rank_cnn = W_HGNET * rank_hgnet + W_EFF * rank_eff + W_EFF10 * rank_eff10 + W_EFB3 * rank_ef3
df_rank_cnn = pd.DataFrame(rank_cnn, columns=CLASSES)
df_rank_cnn.insert(0, "row_id", test_ss_segs)
display(df_rank_cnn.head())

# Protossm

In [ ]:
import sys
import soundfile as sf
ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")
if ONNX_WHL.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    print("ONNX Runtime installed")
import onnxruntime as ort

SR             = 32_000
WINDOW_SEC     = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES   = 60 * SR
N_WINDOWS      = 12
BATCH_FILES    = 16
N_CLASSES      = 234
N_SITES_CAP    = 20
LAMBDA_PRIOR   = 0.4
ENSEMBLE_W     = 0.5
TTA_SHIFTS     = [0, 1, -1, 2, -2]
CONF_POWER     = 0.4
RANK_POWER     = 0.4
DELTA_ALPHA    = 0.20
_WALL_START = time.time()

BASE      = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
ARTIFACT  = Path("/kaggle/input/datasets/jakkma/protossm-pretrained-weights")

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2*d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv-1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state+1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2))
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)
        h = torch.zeros(B_sz, D, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            ys.append((h * C[:, t, None, :]).sum(-1))
        return torch.stack(ys, dim=1) + x * self.D[None, None, :]

class LightProtoSSM(nn.Module):
    def __init__(self, d_input=1536, d_model=256, d_state=32, n_classes=234, n_windows=12,
                 n_ssm_layers=3, dropout=0.15, n_sites=20, meta_dim=16, 
                 use_cross_attn=True, cross_attn_heads=4):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn
        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2*meta_dim, d_model)
        self.ssm_fwd = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_bwd = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(n_ssm_layers)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2*d_model, d_model) for _ in range(n_ssm_layers)])
        self.ssm_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.drop = nn.Dropout(dropout)
        if use_cross_attn:
            self.cross_attn = nn.ModuleList([
                nn.MultiheadAttention(d_model, cross_attn_heads, dropout=dropout, batch_first=True)
                for _ in range(n_ssm_layers)
            ])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(n_ssm_layers)])
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))
    def forward(self, emb, perch_logits=None, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([self.site_emb(site_ids), self.hour_emb(hours)], dim=-1))
            h = h + meta[:, None, :]
        for i, (fwd, bwd, merge, norm) in enumerate(zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm)):
            res = h
            hf = fwd(h)
            hb = bwd(h.flip(1)).flip(1)
            h = self.drop(merge(torch.cat([hf, hb], dim=-1)))
            h = norm(h + res)
            if self.use_cross_attn:
                attn_out, _ = self.cross_attn[i](h, h, h)
                h = self.cross_norm[i](h + attn_out)
        h_n = F.normalize(h, dim=-1)
        p_n = F.normalize(self.prototypes, dim=-1)
        sim = torch.matmul(h_n, p_n.T) * F.softplus(self.proto_temp) + self.class_bias[None, None, :]
        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            out = alpha * sim + (1 - alpha) * perch_logits
        else:
            out = sim
        return out

class ResidualSSM(nn.Module):
    def __init__(self, d_input=1536, d_scores=234, d_model=64, d_state=8, n_classes=234,
                 n_windows=12, dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.n_classes = n_classes
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2*meta_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2*d_model, d_model)
        self.ssm_norm = nn.LayerNorm(d_model)
        self.ssm_drop = nn.Dropout(dropout)
        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)
    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        x = torch.cat([emb, first_pass], dim=-1)
        h = self.input_proj(x) + self.pos_enc[:, :T, :]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([
                self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1)),
                self.hour_emb(hours.clamp(0, 23))
            ], dim=-1))
            h = h + meta.unsqueeze(1)
        res = h
        hf = self.ssm_fwd(h)
        hb = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.ssm_drop(self.ssm_merge(torch.cat([hf, hb], dim=-1)))
        h = self.ssm_norm(h + res)
        return self.output_head(h)

class VectorizedMLPProbes(nn.Module):
    def __init__(self, probe_models):
        super().__init__()
        self.valid_classes = sorted(probe_models.keys())
        V = len(self.valid_classes)
        if V == 0:
            self.weights = nn.ParameterList()
            self.biases = nn.ParameterList()
            self.n_layers = 0
            return
        sample = probe_models[self.valid_classes[0]]
        self.n_layers = len(sample.coefs_)
        self.weights = nn.ParameterList()
        self.biases = nn.ParameterList()
        for li in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], axis=0)
            b = np.stack([probe_models[c].intercepts_[li] for c in self.valid_classes], axis=0)
            self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append(nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False))
    def forward(self, x):
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1:
                h = torch.relu(h)
        return h.squeeze(-1)
        
def _strip_module_prefix(state_dict):
    new_state = {}
    for k, v in state_dict.items():
        if k == "n_averaged":
            continue
        if k.startswith("module."):
            k = k[7:]
        new_state[k] = v
    return new_state
    
print("Loading pretrained weights")
proto_models = []
for seed in [42, 123, 777]:
    m = LightProtoSSM(n_classes=N_CLASSES, n_sites=N_SITES_CAP, use_cross_attn=True, cross_attn_heads=2)
    raw_state = torch.load(ARTIFACT / f"proto_ssm_seed{seed}.pt", map_location="cpu")
    m.load_state_dict(_strip_module_prefix(raw_state))
    m.eval()
    proto_models.append(m)
print(f"Loaded {len(proto_models)} ProtoSSM models")
with open(ARTIFACT / "site2i.json") as f:
    site2i_tr = json.load(f)
print("Loaded site2i mapping")
mlp_vec = torch.load(ARTIFACT / "mlp_probes_vectorized.pt", map_location="cpu", weights_only=False)
mlp_vec.eval()
print("Loaded MLP probes")
import pickle
scaler = pickle.load(open(ARTIFACT / "scaler.pkl", "rb"))
pca    = pickle.load(open(ARTIFACT / "pca.pkl", "rb"))
alpha_blend = float(np.load(ARTIFACT / "alpha_blend.npy"))
print("Loaded scaler, PCA, alpha_blend")
res_ckpt = torch.load(ARTIFACT / "residual_ssm.pt", map_location="cpu")
res_model = ResidualSSM(n_classes=N_CLASSES)
res_model.load_state_dict(_strip_module_prefix(res_ckpt["state_dict"]))
res_model.eval()
correction_weight = res_ckpt["correction_weight"]
print("Loaded ResidualSSM")
_prior = np.load(ARTIFACT / "prior_tables.npz", allow_pickle=True)
prior_tables = {
    "global_p":  _prior["global_p"],
    "site_to_i": _prior["site_to_i"].item(),
    "site_p":    _prior["site_p"],
    "site_n":    _prior["site_n"],
    "hour_to_i": _prior["hour_to_i"].item(),
    "hour_p":    _prior["hour_p"],
    "hour_n":    _prior["hour_n"],
    "sh_to_i":   _prior["sh_to_i"].item(),
    "sh_p":      _prior["sh_p"],
    "sh_n":      _prior["sh_n"],
}
print("Loaded prior tables")
PER_CLASS_THRESHOLDS = np.load(ARTIFACT / "per_class_thresholds.npy")
print("Loaded per-class thresholds")
temperatures = np.load(ARTIFACT / "temperatures.npy")
print("Loaded temperatures")

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))
    
FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")
def parse_fname(name):
    m = FNAME_RE.match(name)
    if not m:
        return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y

def run_perch(paths, batch_files=BATCH_FILES, verbose=True):
    paths = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS
    row_ids   = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites     = np.empty(n_rows, dtype=object)
    hours     = np.zeros(n_rows, dtype=np.int16)
    scores    = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs      = np.zeros((n_rows, 1536), dtype=np.float32)
    wr = 0
    itr = tqdm(range(0, len(paths), batch_files), desc="Perch") if verbose else range(0, len(paths), batch_files)
    import concurrent.futures
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as io_executor:
        next_paths = paths[0:batch_files]
        future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
        for start in itr:
            batch_paths = next_paths
            batch_n = len(batch_paths)
            batch_audio = [f.result() for f in future_audio]
            next_start = start + batch_files
            if next_start < len(paths):
                next_paths = paths[next_start:next_start + batch_files]
                future_audio = [io_executor.submit(read_60s, p) for p in next_paths]
            x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
            br = wr
            for bi, path in enumerate(batch_paths):
                y = batch_audio[bi]
                meta = parse_fname(path.name)
                stem = path.stem
                x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
                row_ids[wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
                filenames[wr:wr + N_WINDOWS] = path.name
                sites[wr:wr + N_WINDOWS] = meta["site"]
                hours[wr:wr + N_WINDOWS] = meta["hour_utc"]
                wr += N_WINDOWS
            outs = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
            logits = outs[ONNX_OUT_MAP["label"]].astype(np.float32)
            emb = outs[ONNX_OUT_MAP["embedding"]].astype(np.float32)
            scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
            embs[br:wr] = emb
            for pos_idx, bc_idxs in proxy_map.items():
                bc_arr = np.array(bc_idxs, dtype=np.int32)
                scores[br:wr, pos_idx] = logits[:, bc_arr].max(axis=1)
            del x, logits, emb, batch_audio
            gc.collect()
    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames, "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

def apply_prior(scores, sites, hours, tables, lambda_prior=LAMBDA_PRIOR):
    eps = 1e-4
    n = len(scores)
    out = scores.copy()
    p = np.tile(tables["global_p"], (n, 1))
    for i, h in enumerate(hours):
        h = int(h)
        if h in tables["hour_to_i"]:
            j = tables["hour_to_i"][h]
            nh = tables["hour_n"][j]
            w = nh / (nh + 8.0)
            p[i] = w * tables["hour_p"][j] + (1 - w) * tables["global_p"]
    for i, s in enumerate(sites):
        s = str(s)
        if s in tables["site_to_i"]:
            j = tables["site_to_i"][s]
            ns = tables["site_n"][j]
            w = ns / (ns + 8.0)
            p[i] = w * tables["site_p"][j] + (1 - w) * p[i]
    if "sh_to_i" in tables:
        for i, (s, h) in enumerate(zip(sites, hours)):
            key = (str(s), int(h))
            if key in tables["sh_to_i"]:
                j = tables["sh_to_i"][key]
                nsh = tables["sh_n"][j]
                w = nsh / (nsh + 4.0)
                p[i] = w * tables["sh_p"][j] + (1 - w) * p[i]
    p = np.clip(p, eps, 1 - eps)
    out += lambda_prior * (np.log(p) - np.log1p(-p))
    return out.astype(np.float32)
def apply_mlp_probes_vectorized(emb_test, scores_test, mlp_vec, scaler, pca, alpha_blend=0.4):
    if len(mlp_vec.valid_classes) == 0:
        return scores_test.copy()
    Z_test = pca.transform(scaler.transform(emb_test)).astype(np.float32)
    valid_classes = mlp_vec.valid_classes
    V = len(valid_classes)
    N = len(scores_test)
    raw = scores_test[:, valid_classes].T
    n_files = N // N_WINDOWS
    raw_view = raw.reshape(V, n_files, N_WINDOWS)
    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt  = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1)
    mx   = np.repeat(raw_view.max(axis=2),  N_WINDOWS, axis=1)
    std  = np.repeat(raw_view.std(axis=2),  N_WINDOWS, axis=1)
    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)
    Z_expanded = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))
    X_all = np.concatenate([Z_expanded.astype(np.float32), scalar_feats], axis=-1)
    with torch.no_grad():
        preds = mlp_vec(torch.tensor(X_all)).numpy()
    result = scores_test.copy()
    result[:, valid_classes] = (1.0 - alpha_blend) * scores_test[:, valid_classes] + alpha_blend * preds.T
    return result
def run_tta_proto(proto_model, emb_files, sc_files, site_t, hour_t, shifts=TTA_SHIFTS):
    proto_model.eval()
    all_preds = []
    emb_t = torch.tensor(emb_files, dtype=torch.float32)
    sc_t  = torch.tensor(sc_files, dtype=torch.float32)
    for shift in shifts:
        e = torch.roll(emb_t, shift, dims=1) if shift else emb_t
        s = torch.roll(sc_t,  shift, dims=1) if shift else sc_t
        with torch.no_grad():
            out = proto_model(e, s, site_ids=site_t, hours=hour_t).numpy()
        if shift:
            out = np.roll(out, -shift, axis=1)
        all_preds.append(out)
    return np.mean(all_preds, axis=0)
def file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=CONF_POWER):
    N, C = probs.shape
    view = probs.reshape(-1, n_windows, C)
    sorted_v = np.sort(view, axis=1)
    top_k_mean = sorted_v[:, -top_k:, :].mean(axis=1, keepdims=True)
    return (view * np.power(top_k_mean, power)).reshape(N, C)
def rank_aware_scaling(probs, n_windows=N_WINDOWS, power=RANK_POWER):
    N, C = probs.shape
    view = probs.reshape(-1, n_windows, C)
    file_max = view.max(axis=1, keepdims=True)
    return (view * np.power(file_max, power)).reshape(N, C)
def adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=DELTA_ALPHA):
    N, C = probs.shape
    result = probs.copy()
    view = probs.reshape(-1, n_windows, C)
    out = result.reshape(-1, n_windows, C)
    for t in range(n_windows):
        conf = view[:, t, :].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        if t == 0:
            neighbor_avg = (view[:, t, :] + view[:, t+1, :]) / 2.0
        elif t == n_windows - 1:
            neighbor_avg = (view[:, t-1, :] + view[:, t, :]) / 2.0
        else:
            neighbor_avg = (view[:, t-1, :] + view[:, t+1, :]) / 2.0
        out[:, t, :] = (1.0 - alpha) * view[:, t, :] + alpha * neighbor_avg
    return out.reshape(N, C)
def apply_per_class_thresholds(scores, thresholds):
    C = scores.shape[1]
    scaled = np.copy(scores)
    for c in range(C):
        t = thresholds[c]
        above = scores[:, c] > t
        scaled[above, c] = 0.5 + 0.5 * (scores[above, c] - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0.0, 1.0)

sample_sub = pd.read_csv(BASE / "sample_submission.csv")
PRIMARY_LABELS = sample_sub.columns[1:].tolist()
assert len(PRIMARY_LABELS) == N_CLASSES, f"N_CLASSES mismatch: {len(PRIMARY_LABELS)} vs {N_CLASSES}"
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
bc_labels = (pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
             .reset_index()
             .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"}))
NO_LABEL = len(bc_labels)
mapping = taxonomy.merge(bc_labels.rename(columns={"scientific_name": "scientific_name"}), on="scientific_name", how="left")
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES    = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK   = BC_INDICES != NO_LABEL
MAPPED_POS    = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)
UNMAPPED_POS  = np.where(~MAPPED_MASK)[0].astype(np.int32)
train_meta_path = ARTIFACT / "train_meta.npz"
if train_meta_path.exists():
    _train_meta = np.load(train_meta_path, allow_pickle=True)
    _saved_labels = _train_meta["primary_labels"].tolist()
    if _saved_labels != PRIMARY_LABELS:
        raise RuntimeError(
            f"PRIMARY_LABELS mismatch: sample_sub has {len(PRIMARY_LABELS)} labels, "
            f"train_meta.npz has {len(_saved_labels)}. Columns may be misaligned!"
        )
    BC_INDICES    = _train_meta["bc_indices"].astype(np.int32)
    MAPPED_MASK   = _train_meta["mapped_mask"].astype(bool)
    MAPPED_POS    = _train_meta["mapped_pos"].astype(np.int32)
    MAPPED_BC_IDX = _train_meta["mapped_bc_idx"].astype(np.int32)
    UNMAPPED_POS  = _train_meta["unmapped_pos"].astype(np.int32)
    print("Loaded exact training mappings from train_meta.npz")
else:
    print("Warning: train_meta.npz not found — using runtime-computed mappings (may differ from training)")

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()
proxy_map = {}
unmapped_df = taxonomy[taxonomy["primary_label"].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].copy()
for _, row in unmapped_df.iterrows():
    target = row["primary_label"]
    sci = str(row["scientific_name"])
    genus = sci.split()[0]
    hits = bc_labels[bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)]
    if len(hits) > 0:
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()
PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}
print(f"Mapped: {MAPPED_MASK.sum()} / {N_CLASSES} species have a Perch logit")
print(f"Unmapped: {len(UNMAPPED_POS)} | Proxy: {len(proxy_map)} | No signal: {len(UNMAPPED_POS)-len(proxy_map)}")
INPUT_ROOT = Path("/kaggle/input")
ONNX_PERCH_PATH = next(INPUT_ROOT.glob("**/perch_v2_no_dft*.onnx"),
                       next(INPUT_ROOT.glob("**/perch_v2*.onnx"), Path("")))
assert ONNX_PERCH_PATH.exists(), "ONNX Perch model not found!"
_so = ort.SessionOptions()
_so.intra_op_num_threads = 4
ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=_so,
                                    providers=["CPUExecutionProvider"])
ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
ONNX_OUT_MAP    = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
print(f"Using ONNX Perch: {ONNX_PERCH_PATH.name}")

test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    test_paths = sorted((BASE / "train_soundscapes").iterdir())[:K]
    print(f"Dry-run on {len(test_paths)} train files")
else:
    print(f"Hidden test files: {len(test_paths)}")
meta_te, sc_te, emb_te = run_perch(test_paths, BATCH_FILES, verbose=True)
print(f"Test scores: {sc_te.shape}  embs: {emb_te.shape}")

n_test_files = len(sc_te) // N_WINDOWS
emb_te_f = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f  = sc_te.reshape(n_test_files, N_WINDOWS, -1)
test_fnames = meta_te.drop_duplicates("filename")["filename"].tolist()
test_site_ids = np.array([
    min(site2i_tr.get(meta_te.loc[meta_te["filename"] == fn, "site"].iloc[0], 0), N_SITES_CAP - 1)
    for fn in test_fnames
], dtype=np.int64)
test_hour_ids = np.array([
    int(meta_te.loc[meta_te["filename"] == fn, "hour_utc"].iloc[0]) % 24
    for fn in test_fnames
], dtype=np.int64)

proto_scores_all_seeds = []
for proto_model in proto_models:
    proto_te = run_tta_proto(
        proto_model, emb_te_f, sc_te_f,
        site_t=torch.tensor(test_site_ids, dtype=torch.long),
        hour_t=torch.tensor(test_hour_ids, dtype=torch.long),
        shifts=TTA_SHIFTS
    )
    proto_scores_all_seeds.append(proto_te.reshape(-1, N_CLASSES).astype(np.float32))

proto_scores_flat = np.mean(proto_scores_all_seeds, axis=0).astype(np.float32)
print("ProtoSSM inference done")

sc_te_adjusted = apply_prior(
    sc_te, sites=meta_te["site"].to_numpy(), hours=meta_te["hour_utc"].to_numpy(),
    tables=prior_tables, lambda_prior=LAMBDA_PRIOR
)
sc_te_adjusted = apply_mlp_probes_vectorized(
    emb_te, sc_te_adjusted, mlp_vec, scaler, pca, alpha_blend
)
first_pass_flat = (ENSEMBLE_W * proto_scores_flat + (1.0 - ENSEMBLE_W) * sc_te_adjusted)
first_pass_te_f = first_pass_flat.reshape(n_test_files, N_WINDOWS, -1)
with torch.no_grad():
    test_correction = res_model(
        torch.tensor(emb_te_f, dtype=torch.float32),
        torch.tensor(first_pass_te_f, dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours=torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()

correction_flat = test_correction.reshape(-1, N_CLASSES).astype(np.float32)
final_scores = first_pass_flat + correction_weight * correction_flat
final_scores = final_scores / temperatures[None, :]
probs = sigmoid(final_scores)
probs = file_confidence_scale(probs, n_windows=N_WINDOWS, top_k=2, power=CONF_POWER)
probs = rank_aware_scaling(probs, n_windows=N_WINDOWS, power=RANK_POWER)
probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=DELTA_ALPHA)
probs = np.clip(probs, 0.0, 1.0)
probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)

df_perch = pd.DataFrame(probs.astype(np.float32), columns=PRIMARY_LABELS)
df_perch.insert(0, "row_id", meta_te["row_id"].values)
df_perch.to_csv("submission_ssm.csv", index=False)
print(f"Total wall time: {(time.time() - _WALL_START)/60:.1f} min")
display(df_perch.head())

In [ ]:
import gc
print("Cleaning up memory for subsequent models")
try:
    del proto_models
    del res_model
    del mlp_vec
    del ONNX_SESSION
except NameError:
    pass
try:
    del sc_te, emb_te, emb_te_f, sc_te_f
    del proto_scores_all_seeds, proto_scores_flat
    del sc_te_adjusted, first_pass_flat, first_pass_te_f
    del test_correction, correction_flat, final_scores, probs
    del meta_te
    del pca, scaler
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("Memory cleanup finished")

# Public sed

In [ ]:
import librosa
from scipy.ndimage import gaussian_filter1d
N_MELS_SED = 256
N_FFT_SED  = 2048
HOP_SED    = 512
FMIN_SED   = 20
FMAX_SED   = 16000
TOP_DB_SED = 80
def find_sed_dir():
    hits = sorted(Path("/kaggle/input").rglob("sed_fold0.onnx"))
    if not hits:
        raise FileNotFoundError("sed_fold0.onnx not found. Attach tuckerarrants/bc2026-distilled-sed-public.")
    return hits[0].parent
def make_sed_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    return ort.InferenceSession(str(path), sess_options=so, providers=["CPUExecutionProvider"])
def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(y=x, sr=SR, n_fft=N_FFT_SED, hop_length=HOP_SED,
                                            n_mels=N_MELS_SED, fmin=FMIN_SED, fmax=FMAX_SED, power=2.0)
        s = librosa.power_to_db(s, top_db=TOP_DB_SED)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)
def file_to_sed_chunks(path):
    y, sr0 = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2: y = y.mean(axis=1)
    if sr0 != SR: y = librosa.resample(y, orig_sr=sr0, target_sr=SR)
    n = 60 * SR
    if len(y) < n: y = np.pad(y, (0, n - len(y)))
    else:          y = y[:n]
    chunks = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    ends   = np.arange(1, N_WINDOWS + 1) * WINDOW_SEC
    return chunks, ends
def sigmoid_sed(x):
    return (1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))).astype(np.float32)
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:K]
sed_dir = find_sed_dir()
sed_fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"),
                         key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
sed_sessions = [make_sed_session(p) for p in sed_fold_paths]
print(f"SED dir: {sed_dir}")
print(f"SED folds loaded: {[p.name for p in sed_fold_paths]}")
sed_rows, sed_preds = [], []
for i, path in enumerate(test_paths, 1):
    chunks, ends = file_to_sed_chunks(path)
    mel = audio_to_mel(chunks)
    p_sum = np.zeros((len(chunks), N_CLASSES), dtype=np.float32)
    for sess in sed_sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})
        clip_logits = outs[0]             # (12, 234)
        frame_max   = outs[1].max(axis=1) # (12, 234)
        p_sum += 0.5 * sigmoid_sed(clip_logits) + 0.5 * sigmoid_sed(frame_max)
    p_mean = p_sum / len(sed_sessions)
    if len(p_mean) > 1:
        p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)
    stem = path.stem
    sed_rows.extend([f"{stem}_{int(t)}" for t in ends])
    sed_preds.append(p_mean)
    if i == 1 or i % 50 == 0 or i == len(test_paths):
        print(f"SED: {i}/{len(test_paths)}")
sed_preds_arr = np.concatenate(sed_preds, axis=0)
df_sed = pd.DataFrame(np.clip(sed_preds_arr, 0.0, 1.0), columns=PRIMARY_LABELS)
df_sed.insert(0, "row_id", sed_rows)
df_sed.to_csv("submission_sed.csv", index=False)
display(df_sed.head())
print(f"Distilled SED Processing Complete. Shape: {df_sed.shape}")
del sed_sessions, sed_preds, sed_preds_arr
gc.collect()

In [ ]:
cols = [c for c in df_perch.columns if c != "row_id"]
row_ids = df_perch["row_id"].values
EPS = 1e-5
df_sed = df_sed.set_index("row_id").loc[df_perch["row_id"]].reset_index()
assert cols == [c for c in df_sed.columns if c != "row_id"]
p_proto = np.clip(df_perch[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
p_sed   = np.clip(df_sed[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
rank_proto = pd.DataFrame(p_proto).rank(axis=0, pct=True).to_numpy(np.float32)
rank_sed   = pd.DataFrame(p_sed).rank(axis=0, pct=True).to_numpy(np.float32)
PROTO_W, SED_W = 0.600, 0.400
pred = PROTO_W * rank_proto + SED_W * rank_sed

# Gate 1
fake_only = (p_proto > 0.50) & (p_sed < 0.05)
pred = np.where(fake_only, (1.0 - 0.08) * pred + 0.08 * rank_proto, pred)

# Gate 2
file_ids = np.array(["_".join(r.split("_")[:-1]) for r in row_ids])
pa_ctx = p_proto.copy()
offs = np.arange(-3, 4, dtype=np.float32)
proto_kernel = (1.0 + (offs / 1.20) ** 2 / 2.0) ** (-1.5)
proto_kernel = (proto_kernel / proto_kernel.sum()).astype(np.float32)
for fid in pd.unique(file_ids):
    m = file_ids == fid
    x = p_proto[m]
    if len(x) > 1:
        xp = np.pad(x, ((3, 3), (0, 0)), mode="edge")
        pa_ctx[m] = sum(proto_kernel[i] * xp[i:i + len(x)] for i in range(7))
xctx = pd.DataFrame(pa_ctx).rank(axis=0, pct=True).to_numpy(np.float32)
proto_cont = (xctx > 0.88) & (rank_proto > 0.75) & (p_sed < 0.12) & (~fake_only)
pred = np.where(proto_cont, (1.0 - 0.15) * pred + 0.15 * np.maximum(rank_proto, xctx), pred)

# Gate 3
sed_only = (rank_sed > 0.95) & (rank_proto < 0.80) & (~fake_only) & (~proto_cont)
pred = np.where(sed_only, (1.0 - 0.12) * pred + 0.12 * rank_sed, pred)

df_rank_perch_sed = pd.DataFrame(pred, columns=cols)
df_rank_perch_sed.insert(0, "row_id", row_ids)
display(df_rank_perch_sed.head())

# ensemble

In [ ]:
cols = [c for c in df_rank_perch_sed.columns if c != "row_id"]
row_ids = df_rank_perch_sed["row_id"].values
EPS = 1e-5
df_rank_cnn   = df_rank_cnn.set_index("row_id").loc[df_rank_perch_sed["row_id"]].reset_index()
assert cols == [c for c in df_rank_cnn.columns if c != "row_id"]
df_rank_perch_sed = np.clip(df_rank_perch_sed[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
df_rank_cnn = np.clip(df_rank_cnn[cols].to_numpy(np.float32), EPS, 1.0 - EPS)
print(f"Perch: {df_rank_perch_sed.shape} Cnn: {df_rank_cnn.shape}")

final_pred = 0.4 * df_rank_perch_sed + 0.6 * df_rank_cnn
final_pred = np.clip(final_pred, EPS, 1.0-EPS).astype(np.float32)
print(f"Final blend: mean={final_pred.mean():.4f}  max={final_pred.max():.4f}")
sub = pd.DataFrame(final_pred, columns=cols)
sub.insert(0, "row_id", row_ids)
col_to_idx = {l: i for i, l in enumerate(cols)}

# 4 Sonotype mirroring
MIRROR_PAIRS = (
    ("47158son15", "47158son16"),
    ("47158son09", "47158son12"),
    ("47158son02", "47158son14"),
    ("47158son13", "47158son21", "47158son22", "47158son23"),
)
mirror_count = 0
for group in MIRROR_PAIRS:
    valid_idx = [col_to_idx[s] for s in group if s in col_to_idx]
    if len(valid_idx) >= 2:
        group_max = sub[cols].iloc[:, valid_idx].max(axis=1).to_numpy(np.float32)
        for idx in valid_idx:
            sub.iloc[:, idx + 1] = group_max
        mirror_count += len(valid_idx)
        
# 5 Rare-class suppression
try:
    tax_df = pd.read_csv(BASE / "taxonomy.csv").set_index("primary_label")
    rare_classes = {"Amphibia", "Mammalia", "Reptilia"}
    rare_count = 0
    for ci, species in enumerate(cols):
        if species in tax_df.index and tax_df.loc[species, "class_name"] in rare_classes:
            col_idx = ci + 1
            vals = sub.iloc[:, col_idx].to_numpy(np.float32)
            thr = vals.mean() + 0.05
            sub.iloc[:, col_idx] = np.where(vals < thr, vals * 0.9, vals)
            rare_count += 1
    print(f"Rare-class suppression 应用于 {rare_count} 列")
except Exception as e:
    print(f"Rare-class 抑制跳过: {e}")

test_paths = list(BASE.glob("test_soundscapes/*.ogg"))
IS_DRY_RUN = len(test_paths) == 0
if IS_DRY_RUN:
    sample_public = pd.read_csv(BASE / "sample_submission.csv")
    template = sub[cols].mean(axis=0).astype(np.float32)
    sub_aligned = sample_public.copy()
    for label in cols:
        sub_aligned[label] = template[label]
    sub = sub_aligned
sub.to_csv("submission.csv", index=False)
print(f"融合完成输出submission.csv|shape={sub.shape}")
display(sub.head())